# 02 - Image inventory

Per dry-season-year (1985-2025): rank candidates by coverage, recommend a single scene or a minimal gap-fill composite, review year-by-year with persistent approvals, and assemble the full inventory. Imagery: Landsat 4/5/7/8/9 + Sentinel-2.

In [ ]:
import os
REPO_DIR = '/content/Shoreline-Dynamics'
# Clone-or-pull so Colab always runs the latest committed code.
if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull -q
else:
    !git clone https://github.com/SKPrince1911/Shoreline-Dynamics.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q earthengine-api geemap

In [ ]:
import ee, geemap
ee.Authenticate()
ee.Initialize(project='shoreline-analysis')
print('GEE initialized:', ee.String('ok').getInfo())

In [ ]:
import sys; sys.path.append('.')
import importlib
import pandas as pd
from src import config, data
importlib.reload(config)  # pick up freshly pulled code (config before data)
importlib.reload(data)

aoi = ee.Geometry.Polygon(config.aoi_coordinates())

def candidates_df(year):
    """Per sensor-date candidates for a dry-season-year (carries season_label)."""
    return data.candidates_dataframe(year, aoi)

def best_df(year):
    """The single select_best pick (or gap record) for a dry-season-year."""
    return pd.DataFrame([data.select_best(year, aoi).getInfo().get('properties', {})])

## Candidate inventory (test years)

In [ ]:
candidates_df(1995)

In [ ]:
candidates_df(2010)

In [ ]:
candidates_df(2022)

## Display helpers

In [ ]:
IDX_PALETTE = [
    '#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00', '#ffd92f',
    '#a65628', '#f781bf', '#66c2a5', '#1b9e77', '#7570b3', '#e7298a',
    '#999999', '#000000',
]

def ranked_df(year):
    """All candidates for a year as a DataFrame, coverage-first (rank 0 = top)."""
    fc = data.year_candidates_ranked(year, aoi)
    rows = [f['properties'] for f in fc.getInfo()['features']]
    cols = ['rank', 'season_label', 'date', 'sensor', 'aoi_cloud_pct',
            'aoi_coverage_pct', 'n_scenes', 'slc_off', 'is_complete',
            'qualifies_95', 'qualifies_90', 'image_ids']
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df[cols].sort_values('rank').reset_index(drop=True)
    return df

def view_candidate(year, rank=0):
    """Show one ranked candidate's true-color mosaic with AOI + tidal channels."""
    fc = data.year_candidates_ranked(year, aoi)
    rows = [f['properties'] for f in fc.getInfo()['features']]
    row = next(r for r in rows if r['rank'] == rank)
    print(f"dry_year {year} ({data.season_label(year)}) | rank {rank} of {len(rows)}: "
          f"{row['date']} {row['sensor']} | cloud {row['aoi_cloud_pct']:.2f}% | "
          f"coverage {row['aoi_coverage_pct']:.2f}%")
    img = data.candidate_mosaic_truecolor(year, aoi, rank)
    m = geemap.Map(); m.add_basemap('SATELLITE'); m.centerObject(aoi, 11)
    m.addLayer(img, {}, f'{year} rank {rank} true color')
    aoi_fc = ee.FeatureCollection([ee.Feature(aoi)])
    m.addLayer(aoi_fc.style(color='red', fillColor='00000000', width=2), {}, 'AOI')
    m.addLayer(data.tidal_channels_fc().style(color='cyan', width=2), {}, 'tidal channels')
    return m

def view_composite(year, sensor=None):
    """Show the minimal gap-fill composite with its date legend and provenance."""
    comp = data.season_fill_composite(year, aoi, sensor=sensor)
    meta = comp.toDictionary([
        'contributing_dates', 'contributing_times', 'contributing_coverage_pct',
        'contributing_marginal_gain_pct', 'contributing_cloud_pct',
        'n_dates', 'n_dates_available', 'achieved_coverage_pct',
    ]).getInfo()
    n_sel = int(meta['n_dates']); n_avail = int(meta['n_dates_available'])
    print(f"composite dry_year {year} ({data.season_label(year)}): "
          f"selected {n_sel} of {n_avail} available dates | "
          f"achieved coverage {meta['achieved_coverage_pct']:.2f}% "
          f"(at {config.COVERAGE_SCALE_FINE} m)")
    dates = meta['contributing_dates']; times = meta['contributing_times']
    covs = meta['contributing_coverage_pct']
    gains = meta['contributing_marginal_gain_pct']
    clouds = meta['contributing_cloud_pct']
    print('selected dates (selection order; src_idx 0 = seed) -- legend:')
    for idx in range(n_sel):
        print(f"  src_idx {idx}: {dates[idx]} | {times[idx]} | "
              f"coverage {covs[idx]:.2f}% | marginal gain {gains[idx]:.2f}% | "
              f"cloud {clouds[idx]:.2f}% | colour {IDX_PALETTE[idx % len(IDX_PALETTE)]}")
    m = geemap.Map(); m.add_basemap('SATELLITE'); m.centerObject(aoi, 11)
    m.addLayer(comp, {'bands': ['R', 'G', 'B'], 'min': 0, 'max': 0.3},
               f'{year} composite true color')
    palette = IDX_PALETTE[:max(1, n_sel)]
    m.addLayer(comp.select('src_idx'),
               {'min': 0, 'max': max(0, n_sel - 1), 'palette': palette},
               'src_idx provenance')
    aoi_fc = ee.FeatureCollection([ee.Feature(aoi)])
    m.addLayer(aoi_fc.style(color='red', fillColor='00000000', width=2), {}, 'AOI')
    m.addLayer(data.tidal_channels_fc().style(color='cyan', width=2), {}, 'tidal channels')
    return m

## Persistent human review

Approvals are written to disk on every call so a Colab disconnect never loses review progress. Mount Google Drive if available so the ledger persists across sessions.

In [ ]:
# Optional Drive mount so approvals survive across Colab sessions.
DRIVE_DIR = '/content/drive/MyDrive/shoreline_dynamics'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_DIR, exist_ok=True)
    APPROVALS_PATH = os.path.join(DRIVE_DIR, 'inventory_approvals.csv')
except Exception as exc:
    os.makedirs('outputs', exist_ok=True)
    APPROVALS_PATH = 'outputs/inventory_approvals.csv'
    print('Google Drive not mounted (', type(exc).__name__, ') -> using repo path.')
print('APPROVALS_PATH =', APPROVALS_PATH)

In [ ]:
import datetime

APPROVAL_COLUMNS = [
    'dry_year', 'season_label', 'product_type', 'sensor', 'n_dates', 'dates',
    'timestamps_utc', 'cloud_pct', 'achieved_coverage_pct', 'image_ids',
    'marginal_gains', 'chosen_rank', 'review_status', 'note', 'reviewed_at_utc',
]
DRY_YEARS = list(range(1985, 2026))  # 1985..2025 inclusive (41 dry-season-years)

if os.path.exists(APPROVALS_PATH):
    approvals = pd.read_csv(APPROVALS_PATH)
else:
    approvals = pd.DataFrame(columns=APPROVAL_COLUMNS)
approvals

In [ ]:
from IPython.display import display

def _product_row(prod, review_status, note, chosen_rank=None):
    """Flatten a year_product dict into an approvals-CSV row."""
    return {
        'dry_year': prod['dry_year'], 'season_label': prod['season_label'],
        'product_type': prod['product_type'], 'sensor': prod['sensor'],
        'n_dates': prod['n_dates'], 'dates': ';'.join(prod['dates']),
        'timestamps_utc': ';'.join(prod['timestamps_utc']),
        'cloud_pct': prod['cloud_pct'],
        'achieved_coverage_pct': prod['achieved_coverage_pct'],
        'image_ids': prod['image_ids'],
        'marginal_gains': ';'.join(f'{g:.4f}' for g in prod['marginal_gains']),
        'chosen_rank': chosen_rank,
        'review_status': review_status, 'note': note,
        'reviewed_at_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }

def _save_approval(row):
    """Replace the row for this dry_year and persist to disk immediately."""
    global approvals
    approvals = approvals[approvals['dry_year'] != row['dry_year']]
    approvals = pd.concat([approvals, pd.DataFrame([row], columns=APPROVAL_COLUMNS)],
                          ignore_index=True)
    approvals.to_csv(APPROVALS_PATH, index=False)
    print(f"Saved dry_year {row['dry_year']} ({row['season_label']}) "
          f"[{row['product_type']} / {row['review_status']}] -> {APPROVALS_PATH}")

def review(year):
    """Show the recommended product for a year (single -> candidate, else composite)."""
    prod = data.year_product(year, aoi)
    cloud = prod['cloud_pct']
    cloud_str = f'{cloud:.2f}%' if cloud is not None else 'n/a'
    print(f"=== dry_year {prod['dry_year']} | season {prod['season_label']} ===")
    print(f"recommended: {prod['product_type']} | sensor {prod['sensor']} | "
          f"{prod['n_dates']} of {prod['n_dates_available']} dates | "
          f"achieved {prod['achieved_coverage_pct']:.2f}% coverage | cloud {cloud_str}")
    print('dates:', prod['dates'])
    if prod['product_type'] == 'single':
        display(ranked_df(year))
        display(view_candidate(year, 0))
    elif prod['product_type'] == 'none':
        print('No usable imagery for this dry-season-year.')
    else:
        display(view_composite(year))
    return year

def approve(year, note=''):
    """Approve the recommended product for a year."""
    prod = data.year_product(year, aoi)
    chosen_rank = 0 if prod['product_type'] == 'single' else None
    _save_approval(_product_row(prod, 'approved', note, chosen_rank=chosen_rank))

def approve_single(year, rank, note=''):
    """Override: approve the rank-N single candidate instead of the recommendation."""
    fc = data.year_candidates_ranked(year, aoi)
    rows = [f['properties'] for f in fc.getInfo()['features']]
    r = next(x for x in rows if x['rank'] == rank)
    prod = {
        'dry_year': year, 'season_label': data.season_label(year),
        'product_type': 'single', 'sensor': r['sensor'], 'n_dates': 1,
        'dates': [r['date']], 'timestamps_utc': [r['timestamp_utc']],
        'cloud_pct': r['aoi_cloud_pct'], 'achieved_coverage_pct': r['aoi_coverage_pct'],
        'image_ids': r['image_ids'], 'marginal_gains': [r['aoi_coverage_pct']],
    }
    _save_approval(_product_row(prod, 'approved_override', note, chosen_rank=rank))

def force_composite(year, note=''):
    """Override: approve the greedy composite even if a single was recommended."""
    _save_approval(_product_row(
        data.year_product(year, aoi, force_composite=True), 'approved_override', note))

def reject(year, note):
    """Mark a year as having no acceptable product (even the composite is unusable)."""
    prod = {
        'dry_year': year, 'season_label': data.season_label(year),
        'product_type': 'none', 'sensor': '', 'n_dates': 0, 'dates': [],
        'timestamps_utc': [], 'cloud_pct': None, 'achieved_coverage_pct': 0,
        'image_ids': '', 'marginal_gains': [],
    }
    _save_approval(_product_row(prod, 'rejected', note))

def show_approvals():
    """Show the approvals ledger, product_type counts, and unreviewed-year count."""
    if approvals.empty:
        print('No approvals recorded yet.')
    else:
        display(approvals.sort_values('dry_year').reset_index(drop=True))
        print('counts by product_type:', approvals['product_type'].value_counts().to_dict())
    reviewed = set(int(y) for y in approvals['dry_year'].tolist())
    unreviewed = [y for y in DRY_YEARS if y not in reviewed]
    print(f'unreviewed: {len(unreviewed)} of {len(DRY_YEARS)} years'
          + (f' (next = {unreviewed[0]})' if unreviewed else ' (all reviewed)'))

def next_unreviewed():
    """Review the earliest 1985-2025 dry-year not yet in the approvals CSV."""
    reviewed = set(int(y) for y in approvals['dry_year'].tolist())
    remaining = [y for y in DRY_YEARS if y not in reviewed]
    print(f'{len(remaining)} of {len(DRY_YEARS)} years remain unreviewed.')
    if not remaining:
        print('All 1985-2025 years reviewed.')
        return None
    y = remaining[0]
    print(f'next unreviewed dry_year: {y} ({data.season_label(y)})')
    return review(y)

In [ ]:
# Paged retrieval note: build_inventory calls year_product per year, and each
# year's aggregations go through the chunked (~8/request) getInfo path in
# src/data.py -- no single request evaluates all 41 years at once.
def build_inventory():
    """Assemble the full 1985-2025 inventory: approved rows override the auto product.

    Writes outputs/image_inventory.csv and prints a summary. Note: computing the
    auto product for every unreviewed year is heavy -- run when ready, not casually.
    """
    reviewed = {int(row['dry_year']): row.to_dict() for _, row in approvals.iterrows()}
    rows = []
    for y in DRY_YEARS:
        if y in reviewed:
            rows.append(reviewed[y])
        else:
            prod = data.year_product(y, aoi)
            cr = 0 if prod['product_type'] == 'single' else None
            rows.append(_product_row(prod, 'auto', '', chosen_rank=cr))
    df = pd.DataFrame(rows, columns=APPROVAL_COLUMNS)
    df = df.sort_values('dry_year').reset_index(drop=True)
    os.makedirs('outputs', exist_ok=True)
    df.to_csv('outputs/image_inventory.csv', index=False)
    counts = df['product_type'].value_counts().to_dict()
    flagged = df[df['product_type'].isin(['partial', 'none'])
                 | (df['review_status'] == 'rejected')]['dry_year'].tolist()
    print('=== image inventory 1985-2025 ===')
    print('total years:', len(df))
    print('product_type:', {k: counts.get(k, 0)
                            for k in ['single', 'composite', 'partial', 'none']})
    print('sensor:', df['sensor'].value_counts().to_dict())
    print('gap/partial/rejected years:', flagged)
    print('rows still auto (unreviewed):',
          int((df['review_status'] == 'auto').sum()), 'of', len(df))
    print('written to outputs/image_inventory.csv')
    return df

### Demo -- review year by year (do NOT auto-run all 41)

`show_approvals()` shows progress; `next_unreviewed()` opens the earliest unreviewed year. Review a year, then `approve(year)` / `approve_single(year, rank)` / `force_composite(year)`. Run `build_inventory()` only once reviewing is done (or to snapshot the current auto+approved state).

In [ ]:
show_approvals()

In [ ]:
next_unreviewed()